In [30]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [32]:
# Dosya yolu
df = pd.read_csv("all_data_ratio.csv")

In [34]:
# Yeni feature_map kullanıcı tarafından verildi
feature_map = {
    "uzman": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "diger"],
    "pratisyen": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "ebe", "diger"],
    "asistan": ["yıl", "nufus", "hastane", "yatak", "uzman", "pratisyen", "hemsire", "ebe", "diger"],
    "ebe": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "pratisyen", "diger"],
    "diger": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "uzman"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "yatak": ["yıl", "nufus", "hastane", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"]
}

In [36]:
tahminler = []

for il in df["il_adı"].unique():
    df_il = df[df["il_adı"] == il]
    if df_il[df_il["yıl"] <= 2021].shape[0] < 5:
        continue

    for hedef, features in feature_map.items():
        if hedef not in df_il.columns or any(f not in df_il.columns for f in features):
            continue

        df_target = df_il[features + [hedef]].dropna()
        if df_target.empty or df_target[df_target["yıl"] <= 2021].shape[0] == 0:
            continue

        train = df_target[df_target["yıl"] <= 2021]
        test = df_target[df_target["yıl"] == 2023]
        if test.empty:
            continue

        X_train = train[features]
        y_train = train[hedef]
        X_test = test[features]
        y_test = test[hedef]

        # Model tahminleri
        lr = LinearRegression().fit(X_train, y_train)
        rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

        y_pred_lr = lr.predict(X_test)
        y_pred_rf = rf.predict(X_test)

        tahminler.append({
            "il_adı": il,
            "hedef": hedef,
            "gercek_2023": y_test.values[0],
            "tahmin_2023_lr": y_pred_lr[0],
            "tahmin_2023_rf": y_pred_rf[0]
        })

In [38]:
# CSV'ye kaydet
df_tahmin = pd.DataFrame(tahminler)
df_tahmin.to_csv("sehir_bazli_2023_tahmin_yeni.csv", index=False)

In [46]:
# Türkiye'nin 7 coğrafi bölgesi
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

# Tahmin dataframe'in varsa:
df_tahmin["bolge"] = df_tahmin["il_adı"].apply(bolge_bul)

# Bölgelere göre topla
bolgesel = df_tahmin.groupby(["bolge", "hedef"])[["gercek_2023", "tahmin_2023_lr", "tahmin_2023_rf"]].sum().reset_index()

# CSV olarak kaydet
bolgesel.to_csv("bolge_bazli_2023_tahmin.csv", index=False)

In [68]:
# 1. Dosyayı oku
df = pd.read_csv("sehir_bazli_2023_tahmin_yeni.csv")  # Gerekirse yolunu değiştir

# 2. Yüzdelik doğruluk hesapla
df["lr_yuzde_dog"] = 100 - (np.abs(df["gercek_2023"] - df["tahmin_2023_lr"]) / df["gercek_2023"] * 100)
df["rf_yuzde_dog"] = 100 - (np.abs(df["gercek_2023"] - df["tahmin_2023_rf"]) / df["gercek_2023"] * 100)

# 3. 0-100 arası sınırla
df["lr_yuzde_dog"] = df["lr_yuzde_dog"].clip(0, 100)
df["rf_yuzde_dog"] = df["rf_yuzde_dog"].clip(0, 100)

# 4. Ortalamaları hesapla
lr_ortalama = round(df["lr_yuzde_dog"].mean(), 2)
rf_ortalama = round(df["rf_yuzde_dog"].mean(), 2)

# 5. Sonuçları yazdır
print(f"\nŞehir Doğruluk Ortalamaları (2023):")
print(f" - Linear Regression: {lr_ortalama}%")
print(f" - Random Forest:     {rf_ortalama}%")

# 6. Dosyaya kaydet
df.to_csv("sehir_bazli_2023_tahmin_dogruluk.csv", index=False)


Şehir Doğruluk Ortalamaları (2023):
 - Linear Regression: 67.81%
 - Random Forest:     78.76%


In [70]:
# 1. Dosyayı oku
df = pd.read_csv("bolge_bazli_2023_tahmin.csv")  # Gerekirse yolunu değiştir

# 2. Yüzdelik doğruluk hesapla
df["lr_yuzde_dog"] = 100 - (np.abs(df["gercek_2023"] - df["tahmin_2023_lr"]) / df["gercek_2023"] * 100)
df["rf_yuzde_dog"] = 100 - (np.abs(df["gercek_2023"] - df["tahmin_2023_rf"]) / df["gercek_2023"] * 100)

# 3. 0-100 arası sınırla
df["lr_yuzde_dog"] = df["lr_yuzde_dog"].clip(0, 100)
df["rf_yuzde_dog"] = df["rf_yuzde_dog"].clip(0, 100)

# 4. Ortalamaları hesapla
lr_ortalama = round(df["lr_yuzde_dog"].mean(), 2)
rf_ortalama = round(df["rf_yuzde_dog"].mean(), 2)

# 5. Sonuçları yazdır
print(f"\nBölge Doğruluk Ortalamaları (2023):")
print(f" - Linear Regression: {lr_ortalama}%")
print(f" - Random Forest:     {rf_ortalama}%")

# 6. Dosyaya kaydet
df.to_csv("bolge_bazli_2023_tahmin_dogruluk.csv", index=False)


Bölge Doğruluk Ortalamaları (2023):
 - Linear Regression: 83.93%
 - Random Forest:     84.77%
